# EyeAI AMD Reference Audit and RAG Index — Final

This notebook builds a local FAISS index from **approved AMD reference sections only**.

Workflow:

`discover PDFs → page audit → approved-page validation → chunking → CPU embeddings → FAISS index → retrieval smoke test`

The notebook does not train either RETFound or Qwen.

## 1. Paths and controls

In [ ]:
from pathlib import Path
import importlib
import json
import os
import shutil
import subprocess
import sys

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")
OUTPUT_DIR = Path("/kaggle/working/eyeai_rag_index/v2")
AUDIT_DIR = Path("/kaggle/working/eyeai_rag_audit")
REFERENCE_MANIFEST = REPO_DIR / "knowledge_base/reference_manifest.yaml"

REFERENCE_DOCUMENTS_OVERRIDE = None
EMBEDDING_MODEL_OVERRIDE = None
EMBEDDING_DEVICE = "cpu"
BUILD_INDEX = True
RUN_RETRIEVAL_SMOKE_TEST = True

print("RAG output:", OUTPUT_DIR)
print("Embedding device:", EMBEDDING_DEVICE)

## 2. Clone the repository and install the assistant stack

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-assistant-kaggle.txt")],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
repo_src = str(REPO_DIR / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
importlib.invalidate_caches()
print("Repository and dependencies are ready.")

## 3. Discover the embedding model and downloaded references

In [ ]:
def discover_embedding_model() -> Path:
    if EMBEDDING_MODEL_OVERRIDE:
        path = Path(EMBEDDING_MODEL_OVERRIDE)
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    candidates = sorted({
        path.parent
        for path in Path("/kaggle/input").rglob("config.json")
        if "qwen3_embedding_0_6b" in str(path.parent).lower()
        and (path.parent / "tokenizer_config.json").is_file()
    })
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one Qwen3 embedding model directory, found: {candidates}")
    return candidates[0]


def discover_reference_documents() -> Path:
    if REFERENCE_DOCUMENTS_OVERRIDE:
        path = Path(REFERENCE_DOCUMENTS_OVERRIDE)
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path

    matching_files = []
    for path in Path("/kaggle/input").rglob("*"):
        if not path.is_file() or path.suffix.lower() != ".pdf":
            continue
        lowered = path.name.lower()
        if (
            "agerelated-macular-degeneration-pdf-1837691334853" in lowered
            or "age-related macular degeneration ppp" in lowered
            or "preferred practice pattern" in lowered
        ):
            matching_files.append(path)

    if len(matching_files) < 2:
        raise FileNotFoundError(
            "Attach a Kaggle Dataset containing both NICE and AAO AMD PDF references. "
            f"Detected matching files: {matching_files}"
        )

    common_root = Path(os.path.commonpath([str(path.parent) for path in matching_files]))
    return common_root

EMBEDDING_MODEL_DIR = discover_embedding_model()
DOCUMENTS_ROOT = discover_reference_documents()

print("Embedding model:", EMBEDDING_MODEL_DIR)
print("Approved reference root:", DOCUMENTS_ROOT)
print("Reference manifest:", REFERENCE_MANIFEST)
print("PDF files:")
for path in sorted(DOCUMENTS_ROOT.rglob("*.pdf")):
    print("-", path)

## 4. Run the page-level reference audit

In [ ]:
if AUDIT_DIR.exists():
    shutil.rmtree(AUDIT_DIR)

subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts/build_rag_index.py"),
        "--documents-root", str(DOCUMENTS_ROOT),
        "--embedding-model", str(EMBEDDING_MODEL_DIR),
        "--output-dir", str(AUDIT_DIR),
        "--reference-manifest", str(REFERENCE_MANIFEST),
        "--audit-only",
        "--device", EMBEDDING_DEVICE,
    ],
    check=True,
    cwd=REPO_DIR,
)
print("Reference audit created:", AUDIT_DIR)

In [ ]:
import pandas as pd

audit_df = pd.read_json(AUDIT_DIR / "reference_audit.json")
summary = (
    audit_df.groupby(["source_id", "selected", "selection_reason"], dropna=False)
    .size()
    .reset_index(name="pages")
)
display(summary)

display_columns = [
    "source_id",
    "page",
    "selected",
    "selection_reason",
    "probable_heading",
    "text_preview",
]
display(audit_df[display_columns].sort_values(["source_id", "page"]).reset_index(drop=True))

selected_counts = audit_df[audit_df["selected"]].groupby("source_id").size().to_dict()
for required_source in ["NICE_NG82", "AAO_AMD_PPP"]:
    if selected_counts.get(required_source, 0) == 0:
        raise RuntimeError(
            f"No approved pages were selected for {required_source}. "
            "Review the audit and adjust knowledge_base/reference_manifest.yaml."
        )
print("Selected page counts:", selected_counts)

## 5. Build the approved-section FAISS index

In [ ]:
if BUILD_INDEX:
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    subprocess.run(
        [
            sys.executable,
            "-u",
            str(REPO_DIR / "scripts/build_rag_index.py"),
            "--documents-root", str(DOCUMENTS_ROOT),
            "--embedding-model", str(EMBEDDING_MODEL_DIR),
            "--output-dir", str(OUTPUT_DIR),
            "--reference-manifest", str(REFERENCE_MANIFEST),
            "--device", EMBEDDING_DEVICE,
            "--chunk-characters", "1500",
            "--overlap-characters", "180",
            "--batch-size", "8",
        ],
        check=True,
        cwd=REPO_DIR,
    )
else:
    print("Index build is disabled after the audit stage.")

## 6. Validate the index and inspect selected chunks

In [ ]:
if BUILD_INDEX:
    required = [
        "index.faiss",
        "chunks.json",
        "manifest.json",
        "reference_audit.json",
        "reference_audit.csv",
    ]
    missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Incomplete RAG index: {missing}")

    manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text(encoding="utf-8"))
    chunks = json.loads((OUTPUT_DIR / "chunks.json").read_text(encoding="utf-8"))
    print(json.dumps(manifest, indent=2, ensure_ascii=False))
    display(pd.DataFrame(chunks)[[
        "source_id", "title", "page", "section", "allowed_topics", "text"
    ]].head(12))

## 7. Retrieval smoke test

In [ ]:
if BUILD_INDEX and RUN_RETRIEVAL_SMOKE_TEST:
    from eyeai.assistant.rag import FaissRagIndex

    rag = FaissRagIndex(
        index_dir=OUTPUT_DIR,
        embedding_model_path=EMBEDDING_MODEL_DIR,
        top_k=3,
        minimum_score=0.20,
        maximum_chunk_characters=1800,
        maximum_chunks_per_source=1,
        device="cpu",
    )
    results = rag.search(
        "How should a positive binary AMD screening result be clinically reviewed, and why does it not determine severity?",
        allowed_topics=["classification", "clinical_evaluation", "diagnosis", "clinical_limitations"],
        maximum_chunks_per_source=1,
    )
    if not results:
        raise RuntimeError("The RAG smoke query returned no approved references.")
    for number, item in enumerate(results, start=1):
        print(f"[{number}] {item.source_id} | page={item.page} | section={item.section} | score={item.score:.4f}")
        print(item.text[:700])
        print()

print("Save this notebook with output and attach eyeai_rag_index/v2 to Notebook 15.")